# Malware image classification on MaleVis (binary: malware vs benign)

This notebook answers the design-and-reasoning task as a **report-style workflow**:

1. **EDA first**: scan the MaleVis folder structure, build metadata, show original-label and binary-label counts, and visualize class structure.
2. Convert the original 26-class problem into **binary malware detection** by collapsing all malware families into one class called **`malware`** and keeping the legitimate-software class as **`benign`**.
3. Create a **leakage-aware split strategy** with a visible split/leakage audit.
4. Compare **one lighter baseline** and **one stronger model**.
5. Add a small **preprocessing ablation** (`grayscale` vs `RGB`).
6. Report **confusion matrix**, **precision**, **recall**, **F1-score**, and **balanced accuracy**.
7. Build an **error-analysis table** using at least 8 misclassified images.

## Source notes

MaleVis is described by its official page as an image dataset with **25 malware classes + 1 legitimate software class**, and the official release includes **9100 training** and **5126 validation** RGB images. The Kaggle copy also exposes `train` and `val` folders with **26 directories**.  
This notebook computes all counts directly from your local copy so the visible evidence comes from the actual files you use.

**References used to design this notebook**
- Official MaleVis site: https://web.cs.hacettepe.edu.tr/~selman/malevis/
- Kaggle dataset page: https://www.kaggle.com/datasets/sohamkumar1703/malevis-dataset

## Task / Question
**Import the required libraries, configure plotting colors, and set the random seed.**

In [ ]:

# Optional install commands for Google Colab or a fresh environment:
# !pip install torch torchvision scikit-learn pillow seaborn tqdm pandas matplotlib

import os
import random
import hashlib
from pathlib import Path
from collections import Counter
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

## Task / Question
**Point the notebook to the MaleVis dataset and automatically discover the `train` and `val` folders.**

Update `DATA_ROOT` so it points to your local MaleVis copy from Kaggle or the official download.

In [ ]:

# Update this path to your MaleVis dataset root.
# Expected structure somewhere under DATA_ROOT:
#   .../train/<family_name>/*.png
#   .../val/<family_name>/*.png
DATA_ROOT = Path("/content/malevis_train_val_224x224")  # <-- change this if needed

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

BENIGN_KEYWORDS = {
    "legitimate", "legit", "benign", "clean", "goodware", "other"
}

def find_split_dir(root: Path, split_name: str):
    candidates = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == split_name.lower()]
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not find a '{split_name}' directory under {root}. "
            "Please check DATA_ROOT."
        )
    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    return candidates[0]

train_dir = find_split_dir(DATA_ROOT, "train")
val_dir = find_split_dir(DATA_ROOT, "val")

print("Train directory:", train_dir)
print("Validation directory:", val_dir)

## Task / Question
**Run EDA first: scan the folder structure, build a metadata table, detect the original family labels, convert them to the binary labels, and show a dataset overview table.**

In [ ]:

def family_to_binary_label(family_name: str) -> str:
    """
    Convert the original MaleVis family name into the binary target.
    Any legitimate / benign-like folder name becomes 'benign'.
    All malware families collapse into 'malware'.
    """
    name = family_name.lower().strip()
    if name in BENIGN_KEYWORDS or any(key in name for key in BENIGN_KEYWORDS):
        return "benign"
    return "malware"

def scan_split(split_dir: Path, split_name: str) -> pd.DataFrame:
    rows = []
    class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    for class_dir in sorted(class_dirs):
        family = class_dir.name
        binary = family_to_binary_label(family)
        for img_path in class_dir.rglob("*"):
            if img_path.suffix.lower() in IMAGE_EXTS:
                rows.append({
                    "path": str(img_path),
                    "split_from_disk": split_name,
                    "original_family": family,
                    "binary_label": binary,
                    "file_name": img_path.name
                })
    return pd.DataFrame(rows)

meta_train_disk = scan_split(train_dir, "official_train")
meta_val_disk = scan_split(val_dir, "official_val")
metadata = pd.concat([meta_train_disk, meta_val_disk], ignore_index=True)

if metadata.empty:
    raise ValueError("No image files were found. Check the folder path and image extensions.")

metadata["binary_target_id"] = metadata["binary_label"].map({"benign": 0, "malware": 1})

dataset_overview = pd.DataFrame({
    "Property": [
        "Total images found",
        "Images in official train folder",
        "Images in official val folder",
        "Original family count",
        "Binary classes",
        "Benign images",
        "Malware images"
    ],
    "Value": [
        len(metadata),
        (metadata["split_from_disk"] == "official_train").sum(),
        (metadata["split_from_disk"] == "official_val").sum(),
        metadata["original_family"].nunique(),
        metadata["binary_label"].nunique(),
        (metadata["binary_label"] == "benign").sum(),
        (metadata["binary_label"] == "malware").sum()
    ]
})

display(dataset_overview)

label_mapping_table = (
    metadata[["original_family", "binary_label"]]
    .drop_duplicates()
    .sort_values(["binary_label", "original_family"])
    .reset_index(drop=True)
)

display(label_mapping_table)

## Task / Question
**Report the original-label counts and the resulting binary class counts, then visualize them with colorful EDA plots.**

In [ ]:

original_counts = (
    metadata.groupby(["split_from_disk", "original_family"])
    .size()
    .reset_index(name="count")
)

binary_counts = (
    metadata.groupby(["split_from_disk", "binary_label"])
    .size()
    .reset_index(name="count")
)

display(
    original_counts.pivot(index="original_family", columns="split_from_disk", values="count").fillna(0).astype(int)
)
display(
    binary_counts.pivot(index="binary_label", columns="split_from_disk", values="count").fillna(0).astype(int)
)

plt.figure(figsize=(14, 7))
family_plot_df = original_counts.sort_values(["split_from_disk", "count", "original_family"], ascending=[True, False, True])
sns.barplot(
    data=family_plot_df,
    x="original_family",
    y="count",
    hue="split_from_disk",
    palette="Set2"
)
plt.xticks(rotation=80, ha="right")
plt.title("Original MaleVis family counts by official split")
plt.xlabel("Original family")
plt.ylabel("Image count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(
    data=binary_counts,
    x="binary_label",
    y="count",
    hue="split_from_disk",
    palette="viridis"
)
plt.title("Binary class counts after label conversion")
plt.xlabel("Binary target")
plt.ylabel("Image count")
plt.tight_layout()
plt.show()

## Task / Question
**Show a small visual EDA panel with example images from different original families and both binary targets.**

In [ ]:

def show_sample_images(meta_df: pd.DataFrame, n_per_label: int = 4, image_size=(3.2, 3.2)):
    sampled_rows = []
    for binary_label in ["benign", "malware"]:
        subset = meta_df[meta_df["binary_label"] == binary_label].copy()
        families = subset["original_family"].drop_duplicates().tolist()
        if binary_label == "benign":
            chosen_families = families[:1]
        else:
            chosen_families = families[:n_per_label]
        for fam in chosen_families:
            row = subset[subset["original_family"] == fam].sample(1, random_state=SEED)
            sampled_rows.append(row)
    sampled = pd.concat(sampled_rows, ignore_index=True)

    fig, axes = plt.subplots(1, len(sampled), figsize=(image_size[0] * len(sampled), image_size[1]))
    if len(sampled) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, sampled.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{row['original_family']}\n→ {row['binary_label']}", fontsize=10)
        ax.axis("off")

    plt.suptitle("Sample MaleVis images after binary label mapping", y=1.05, fontsize=14)
    plt.tight_layout()
    plt.show()

show_sample_images(metadata)

## Task / Question
**Give short pseudocode for the binary label conversion and the leakage-aware split logic.**

```text
PSEUDOCODE — LABEL CONVERSION
for each image folder name:
    if folder name corresponds to legitimate / benign software:
        binary_label = "benign"
    else:
        binary_label = "malware"

PSEUDOCODE — SPLIT LOGIC
1. Read the official MaleVis train and val folders.
2. Build metadata with: file path, original family, binary label, and official split.
3. Compute exact file hashes and remove exact duplicates globally before learning.
4. Keep the official val folder untouched as the final test split.
5. Split the official train folder into internal train and internal validation,
   stratifying by original family so the family mix remains visible.
6. Tune preprocessing and model choices on internal validation only.
7. Evaluate the final chosen model once on the untouched test split.
```

## Task / Question
**Design a leakage-aware split strategy, audit exact duplicates, and explain one realistic residual leakage risk.**

The strategy used here is:

- **Official train** → becomes the pool for **internal train + internal validation**
- **Official val** → becomes the untouched **final test split**
- exact duplicate images are removed globally **before** the internal split
- internal train/validation are stratified by the **original family**, not only by the binary label

This reduces leakage, but one **residual leakage risk** still remains: malware images derived from related binaries or visually similar variants can still appear across splits even when exact duplicates are removed.

In [ ]:

def sha1_of_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    hasher = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()

metadata["sha1"] = [sha1_of_file(p) for p in tqdm(metadata["path"], desc="Hashing files")]

train_hashes = set(metadata.loc[metadata["split_from_disk"] == "official_train", "sha1"])
val_hashes = set(metadata.loc[metadata["split_from_disk"] == "official_val", "sha1"])
exact_cross_split_duplicates = len(train_hashes.intersection(val_hashes))

metadata_dedup = metadata.drop_duplicates(subset="sha1").copy()
removed_duplicates = len(metadata) - len(metadata_dedup)

train_pool = metadata_dedup[metadata_dedup["split_from_disk"] == "official_train"].copy()
test_meta = metadata_dedup[metadata_dedup["split_from_disk"] == "official_val"].copy()

train_meta, valid_meta = train_test_split(
    train_pool,
    test_size=0.20,
    random_state=SEED,
    stratify=train_pool["original_family"]
)

train_meta = train_meta.copy()
valid_meta = valid_meta.copy()
test_meta = test_meta.copy()

train_meta["split_final"] = "train"
valid_meta["split_final"] = "validation"
test_meta["split_final"] = "test"

split_meta = pd.concat([train_meta, valid_meta, test_meta], ignore_index=True)

def count_exact_hash_overlap(df_a: pd.DataFrame, df_b: pd.DataFrame) -> int:
    return len(set(df_a["sha1"]).intersection(set(df_b["sha1"])))

split_leakage_audit = pd.DataFrame([
    ["Official train vs official val exact duplicate hashes", exact_cross_split_duplicates, "Before deduplication"],
    ["Images removed by global exact-duplicate deduplication", removed_duplicates, "Before internal splitting"],
    ["Internal train vs validation exact duplicate hashes", count_exact_hash_overlap(train_meta, valid_meta), "After deduplication"],
    ["Internal train vs test exact duplicate hashes", count_exact_hash_overlap(train_meta, test_meta), "After deduplication"],
    ["Internal validation vs test exact duplicate hashes", count_exact_hash_overlap(valid_meta, test_meta), "After deduplication"],
], columns=["Leakage audit item", "Count", "When measured"])

display(split_leakage_audit)

split_count_table = (
    split_meta.groupby(["split_final", "binary_label"])
    .size()
    .reset_index(name="count")
    .pivot(index="split_final", columns="binary_label", values="count")
    .fillna(0)
    .astype(int)
)

display(split_count_table)

## Short interpretation
This split design is leakage-aware because the final test set is held out from the start and exact duplicates are removed **before** the internal train/validation split.  
A realistic **residual leakage risk** still remains: related binaries or visually similar variants may appear across train/validation/test even when their image bytes are not identical. That is much harder to eliminate without stronger grouping information about the underlying binaries.

## Task / Question
**Build the image datasets and preprocessing pipelines. Use the EDA findings to keep the binary target visible and to support two preprocessing choices in the ablation study: grayscale vs RGB.**

In [ ]:

CLASS_TO_ID = {"benign": 0, "malware": 1}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

def build_transform(mode: str = "rgb", image_size: int = 224):
    """
    Keep preprocessing simple and leakage-safe.
    No aggressive geometric augmentation is used here because malware byte-images
    are structure-sensitive and flips/rotations may create unrealistic artifacts.
    """
    if mode == "rgb":
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    elif mode == "grayscale":
        return transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.25])
        ])
    else:
        raise ValueError("mode must be 'rgb' or 'grayscale'")

class MaleVisBinaryDataset(Dataset):
    def __init__(self, meta_df: pd.DataFrame, mode: str = "rgb", image_size: int = 224):
        self.meta = meta_df.reset_index(drop=True).copy()
        self.mode = mode
        self.transform = build_transform(mode=mode, image_size=image_size)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        x = self.transform(img)
        y = int(CLASS_TO_ID[row["binary_label"]])
        return x, y, row["path"], row["original_family"]

def build_loaders(train_df, valid_df, test_df, mode="rgb", image_size=224, batch_size=64, num_workers=2):
    train_ds = MaleVisBinaryDataset(train_df, mode=mode, image_size=image_size)
    valid_ds = MaleVisBinaryDataset(valid_df, mode=mode, image_size=image_size)
    test_ds = MaleVisBinaryDataset(test_df, mode=mode, image_size=image_size)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, valid_loader, test_loader

train_label_counts = train_meta["binary_label"].value_counts().sort_index()
class_weights = torch.tensor(
    [
        len(train_meta) / (2 * train_label_counts["benign"]),
        len(train_meta) / (2 * train_label_counts["malware"])
    ],
    dtype=torch.float32
)

display(pd.DataFrame({
    "Class": ["benign", "malware"],
    "Train count": [train_label_counts["benign"], train_label_counts["malware"]],
    "CrossEntropy class weight": class_weights.numpy()
}))

## Task / Question
**Define one lighter baseline model and one stronger model, plus the shared training and evaluation utilities.**

In [ ]:

class SmallCNN(nn.Module):
    """
    Lighter baseline model.
    Good as a compact reference point and fast enough for the ablation study.
    """
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(192, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def build_resnet18_binary(pretrained=True):
    """
    Stronger model: transfer learning with ResNet18.
    The backbone is mostly frozen to keep runtime manageable and logic clear.
    """
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    except Exception:
        print("Pretrained weights unavailable. Falling back to random initialization.")
        model = models.resnet18(weights=None)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 2)
    return model

def compute_metrics_from_arrays(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)
    }

def evaluate_model(model, loader, criterion):
    model.eval()
    losses = []
    all_probs = []
    all_preds = []
    all_labels = []
    all_paths = []
    all_families = []

    with torch.no_grad():
        for images, labels, paths, families in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)[:, 1]

            preds = (probs >= 0.5).long()

            losses.append(loss.item())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_paths.extend(list(paths))
            all_families.extend(list(families))

    metrics = compute_metrics_from_arrays(all_labels, all_preds)
    metrics["loss"] = float(np.mean(losses))
    metrics["confusion_matrix"] = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    outputs = pd.DataFrame({
        "path": all_paths,
        "original_family": all_families,
        "y_true": all_labels,
        "y_pred": all_preds,
        "prob_malware": all_probs
    })
    outputs["confidence"] = outputs["prob_malware"].where(outputs["y_pred"] == 1, 1 - outputs["prob_malware"])
    outputs["true_label"] = outputs["y_true"].map(ID_TO_CLASS)
    outputs["pred_label"] = outputs["y_pred"].map(ID_TO_CLASS)

    return metrics, outputs

def fit_model(model, train_loader, valid_loader, criterion, optimizer, epochs=6):
    best_state = None
    best_val_f1 = -1.0
    history_rows = []

    model = model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for images, labels, _, _ in tqdm(train_loader, leave=False, desc=f"Epoch {epoch}/{epochs}"):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_metrics, _ = evaluate_model(model, valid_loader, criterion)

        history_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"]
        })

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    history = pd.DataFrame(history_rows)
    final_val_metrics, final_val_outputs = evaluate_model(model, valid_loader, criterion)
    return model, history, final_val_metrics, final_val_outputs

def plot_training_history(history_df: pd.DataFrame, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.lineplot(data=history_df, x="epoch", y="train_loss", marker="o", ax=axes[0], color="#1f77b4")
    sns.lineplot(data=history_df, x="epoch", y="val_loss", marker="o", ax=axes[0], color="#ff7f0e")
    axes[0].set_title(f"{title} — loss")
    axes[0].legend(["train_loss", "val_loss"])

    sns.lineplot(data=history_df, x="epoch", y="val_f1", marker="o", ax=axes[1], color="#2ca02c")
    sns.lineplot(data=history_df, x="epoch", y="val_balanced_accuracy", marker="o", ax=axes[1], color="#d62728")
    axes[1].set_title(f"{title} — validation metrics")
    axes[1].legend(["val_f1", "val_balanced_accuracy"])
    plt.tight_layout()
    plt.show()

## Task / Question
**Run the preprocessing ablation study and compare `grayscale` vs `RGB` using the lighter baseline model.**

This is the requested **small ablation table**.  
The baseline architecture is kept fixed so the comparison isolates the effect of preprocessing choice.

In [ ]:

ABLATION_EPOCHS = 4
BATCH_SIZE = 64

ablation_configs = [
    {"name": "Grayscale 128x128", "mode": "grayscale", "image_size": 128, "in_channels": 1},
    {"name": "RGB 128x128", "mode": "rgb", "image_size": 128, "in_channels": 3},
]

ablation_results = []
ablation_models = {}

for cfg in ablation_configs:
    print(f"\nRunning ablation: {cfg['name']}")
    train_loader, valid_loader, test_loader = build_loaders(
        train_meta, valid_meta, test_meta,
        mode=cfg["mode"],
        image_size=cfg["image_size"],
        batch_size=BATCH_SIZE
    )

    model = SmallCNN(in_channels=cfg["in_channels"]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    trained_model, history, val_metrics, val_outputs = fit_model(
        model, train_loader, valid_loader, criterion, optimizer, epochs=ABLATION_EPOCHS
    )

    ablation_models[cfg["name"]] = {
        "model": trained_model,
        "history": history,
        "val_metrics": val_metrics,
        "val_outputs": val_outputs,
        "config": cfg
    }

    ablation_results.append({
        "Preprocessing": cfg["name"],
        "Validation Precision": val_metrics["precision"],
        "Validation Recall": val_metrics["recall"],
        "Validation F1-score": val_metrics["f1"],
        "Validation Balanced Accuracy": val_metrics["balanced_accuracy"]
    })

ablation_table = pd.DataFrame(ablation_results).sort_values(
    by=["Validation F1-score", "Validation Balanced Accuracy"],
    ascending=False
).reset_index(drop=True)

display(ablation_table.round(4))

plt.figure(figsize=(10, 5))
ablation_plot = ablation_table.melt(
    id_vars="Preprocessing",
    value_vars=["Validation Precision", "Validation Recall", "Validation F1-score", "Validation Balanced Accuracy"],
    var_name="Metric",
    value_name="Score"
)
sns.barplot(data=ablation_plot, x="Preprocessing", y="Score", hue="Metric", palette="Spectral")
plt.ylim(0, 1)
plt.title("Preprocessing ablation using the lighter baseline model")
plt.xticks(rotation=12)
plt.tight_layout()
plt.show()

best_baseline_preprocessing = ablation_table.iloc[0]["Preprocessing"]
print("Chosen preprocessing for the lighter baseline:", best_baseline_preprocessing)

## Short interpretation
The ablation table above is visible evidence for the preprocessing choice.  
The better variant becomes the **final lighter baseline configuration** for the main model comparison.

## Task / Question
**Train the lighter baseline with the chosen preprocessing and compare it against one stronger model. Report both validation and test performance in one compact model-comparison table.**

In [ ]:

baseline_cfg = ablation_models[best_baseline_preprocessing]["config"]

baseline_train_loader, baseline_valid_loader, baseline_test_loader = build_loaders(
    train_meta, valid_meta, test_meta,
    mode=baseline_cfg["mode"],
    image_size=baseline_cfg["image_size"],
    batch_size=BATCH_SIZE
)

baseline_model = SmallCNN(in_channels=baseline_cfg["in_channels"]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)

baseline_model, baseline_history, baseline_val_metrics, baseline_val_outputs = fit_model(
    baseline_model, baseline_train_loader, baseline_valid_loader, criterion, baseline_optimizer, epochs=6
)
baseline_test_metrics, baseline_test_outputs = evaluate_model(baseline_model, baseline_test_loader, criterion)

strong_train_loader, strong_valid_loader, strong_test_loader = build_loaders(
    train_meta, valid_meta, test_meta,
    mode="rgb",
    image_size=224,
    batch_size=48
)

strong_model = build_resnet18_binary(pretrained=True).to(device)
strong_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, strong_model.parameters()),
    lr=1e-4
)

strong_model, strong_history, strong_val_metrics, strong_val_outputs = fit_model(
    strong_model, strong_train_loader, strong_valid_loader, criterion, strong_optimizer, epochs=5
)
strong_test_metrics, strong_test_outputs = evaluate_model(strong_model, strong_test_loader, criterion)

model_comparison_table = pd.DataFrame([
    {
        "Model": f"Lighter baseline — SmallCNN ({baseline_cfg['name']})",
        "Validation Precision": baseline_val_metrics["precision"],
        "Validation Recall": baseline_val_metrics["recall"],
        "Validation F1-score": baseline_val_metrics["f1"],
        "Validation Balanced Accuracy": baseline_val_metrics["balanced_accuracy"],
        "Test Precision": baseline_test_metrics["precision"],
        "Test Recall": baseline_test_metrics["recall"],
        "Test F1-score": baseline_test_metrics["f1"],
        "Test Balanced Accuracy": baseline_test_metrics["balanced_accuracy"]
    },
    {
        "Model": "Stronger model — ResNet18 transfer learning (RGB 224x224)",
        "Validation Precision": strong_val_metrics["precision"],
        "Validation Recall": strong_val_metrics["recall"],
        "Validation F1-score": strong_val_metrics["f1"],
        "Validation Balanced Accuracy": strong_val_metrics["balanced_accuracy"],
        "Test Precision": strong_test_metrics["precision"],
        "Test Recall": strong_test_metrics["recall"],
        "Test F1-score": strong_test_metrics["f1"],
        "Test Balanced Accuracy": strong_test_metrics["balanced_accuracy"]
    }
]).sort_values(by=["Test F1-score", "Test Balanced Accuracy"], ascending=False).reset_index(drop=True)

display(model_comparison_table.round(4))

plot_df = model_comparison_table.melt(
    id_vars="Model",
    value_vars=["Test Precision", "Test Recall", "Test F1-score", "Test Balanced Accuracy"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model", palette="Set1")
plt.ylim(0, 1)
plt.title("Main model comparison on the test split")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

best_model_name = model_comparison_table.iloc[0]["Model"]
best_model = strong_model if "ResNet18" in best_model_name else baseline_model
best_test_outputs = strong_test_outputs if "ResNet18" in best_model_name else baseline_test_outputs
best_val_outputs = strong_val_outputs if "ResNet18" in best_model_name else baseline_val_outputs
best_test_metrics = strong_test_metrics if "ResNet18" in best_model_name else baseline_test_metrics

print("Best test model:", best_model_name)

## Task / Question
**Show one confusion matrix for the best model and report precision, recall, F1-score, and balanced accuracy explicitly.**

In [ ]:

best_cm = best_test_metrics["confusion_matrix"]

metric_table = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1-score", "Balanced Accuracy"],
    "Best model test value": [
        best_test_metrics["precision"],
        best_test_metrics["recall"],
        best_test_metrics["f1"],
        best_test_metrics["balanced_accuracy"]
    ]
})

display(metric_table.round(4))

plt.figure(figsize=(6, 5))
sns.heatmap(
    best_cm,
    annot=True,
    fmt="d",
    cmap="mako",
    xticklabels=["benign", "malware"],
    yticklabels=["benign", "malware"]
)
plt.title(f"Confusion matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

report_df = pd.DataFrame(
    classification_report(
        best_test_outputs["y_true"],
        best_test_outputs["y_pred"],
        target_names=["benign", "malware"],
        output_dict=True
    )
).T

display(report_df.round(4))

## Task / Question
**Build the error-analysis table using at least 8 misclassified images and classify each error into one of the required groups:**
- visually ambiguous pattern
- benign-software confusion
- weak representation
- train/test mismatch
- preprocessing issue

The categorization below uses transparent heuristics so the evidence is visible in the report.

In [ ]:

family_val_summary = (
    best_val_outputs
    .assign(correct=lambda d: (d["y_true"] == d["y_pred"]).astype(int))
    .groupby("original_family")
    .agg(validation_samples=("correct", "size"),
         validation_accuracy=("correct", "mean"))
    .reset_index()
)
family_val_summary["validation_error_rate"] = 1 - family_val_summary["validation_accuracy"]
family_error_threshold = family_val_summary["validation_error_rate"].quantile(0.75)

def image_gray_stats(path: str):
    img = Image.open(path).convert("L")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr.mean(), arr.std()

train_stat_sample = train_meta.sample(min(500, len(train_meta)), random_state=SEED)
train_stat_values = np.array([image_gray_stats(p) for p in tqdm(train_stat_sample["path"], desc="Computing train image stats")])
train_mean_low, train_mean_high = np.quantile(train_stat_values[:, 0], [0.05, 0.95])
train_std_low, train_std_high = np.quantile(train_stat_values[:, 1], [0.05, 0.95])

test_outputs_by_preprocessing = {}

for prep_name, obj in ablation_models.items():
    cfg = obj["config"]
    _, _, temp_test_loader = build_loaders(
        train_meta, valid_meta, test_meta,
        mode=cfg["mode"],
        image_size=cfg["image_size"],
        batch_size=BATCH_SIZE
    )
    temp_metrics, temp_outputs = evaluate_model(obj["model"], temp_test_loader, criterion)
    test_outputs_by_preprocessing[prep_name] = temp_outputs[["path", "y_pred"]].rename(columns={"y_pred": f"pred_{prep_name}"})

misclassified = best_test_outputs[best_test_outputs["y_true"] != best_test_outputs["y_pred"]].copy()

misclassified = misclassified.merge(
    family_val_summary[["original_family", "validation_error_rate"]],
    on="original_family",
    how="left"
)

for prep_name, temp_df in test_outputs_by_preprocessing.items():
    misclassified = misclassified.merge(temp_df, on="path", how="left")

means = []
stds = []
for path in tqdm(misclassified["path"], desc="Computing error image stats"):
    m, s = image_gray_stats(path)
    means.append(m)
    stds.append(s)

misclassified["gray_mean"] = means
misclassified["gray_std"] = stds

def categorize_error(row):
    if row["true_label"] == "benign" or row["pred_label"] == "benign":
        return "benign-software confusion", "One side of the mistake is the benign class."

    preprocessing_columns = [c for c in row.index if c.startswith("pred_")]
    fixed_by_alt = any(row[c] == row["y_true"] for c in preprocessing_columns if pd.notna(row[c]))
    if fixed_by_alt:
        return "preprocessing issue", "An alternative preprocessing setting corrected this sample."

    if (row["gray_mean"] < train_mean_low or row["gray_mean"] > train_mean_high or
        row["gray_std"] < train_std_low or row["gray_std"] > train_std_high):
        return "train/test mismatch", "The image statistics fall outside the central train-range band."

    if row["validation_error_rate"] >= family_error_threshold:
        return "weak representation", "This original family already showed above-average validation error."

    if row["confidence"] < 0.70:
        return "visually ambiguous pattern", "The model is uncertain, suggesting limited visual separation."

    return "visually ambiguous pattern", "Fallback category for a hard visual decision."

assigned = misclassified.apply(categorize_error, axis=1)
misclassified["error_group"] = [x[0] for x in assigned]
misclassified["reason"] = [x[1] for x in assigned]

error_table = (
    misclassified
    .sort_values(["error_group", "confidence"], ascending=[True, False])
    .groupby("error_group", group_keys=False)
    .head(3)
)

if len(error_table) < 8:
    remaining = misclassified.drop(index=error_table.index)
    needed = min(8 - len(error_table), len(remaining))
    if needed > 0:
        extra = remaining.sample(n=needed, random_state=SEED)
        error_table = pd.concat([error_table, extra], ignore_index=True)
else:
    error_table = error_table.head(12)

error_table = error_table.copy()
error_table["file_name"] = error_table["path"].apply(lambda p: Path(p).name)

display_cols = [
    "file_name", "original_family", "true_label", "pred_label",
    "confidence", "validation_error_rate", "error_group", "reason"
]
display(error_table[display_cols].reset_index(drop=True).round(4))

error_summary = (
    misclassified["error_group"]
    .value_counts()
    .rename_axis("error_group")
    .reset_index(name="misclassified_image_count")
)

display(error_summary)

## Task / Question
**Show a small gallery of misclassified images so the error analysis is visible, not hidden in code.**

In [ ]:

def plot_error_gallery(error_df: pd.DataFrame, n: int = 8):
    subset = error_df.head(n).copy()
    n = len(subset)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(rows, cols)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, (_, row) in zip(axes.ravel(), subset.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(
            f"{row['true_label']} → {row['pred_label']}\n{row['error_group']}",
            fontsize=10
        )
        ax.axis("off")

    plt.suptitle("Misclassified-image gallery for the chosen model", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_error_gallery(error_table, n=min(8, len(error_table)))

## Short interpretation
The error table and gallery make the failure modes visible.  
This is important because a high aggregate score can still hide a systematic weakness, especially in a binary malware-detection setting where **benign-vs-malware confusion** is operationally costly.

## Final compact checklist

The notebook now contains the evidence requested in the task:

- **label-mapping table**
- **binary class counts**
- **EDA figures**
- **split / leakage-audit table**
- **short pseudocode**
- **preprocessing-ablation table**
- **lighter baseline vs stronger model comparison**
- **confusion matrix**
- **precision / recall / F1 / balanced accuracy**
- **error-analysis table with at least 8 misclassified images**

You can increase the number of epochs or switch to a larger transfer-learning model if you want a stronger final score, but the current notebook is designed to keep the logic visible and the runtime manageable.